# IK Model Trainer
**Inputs:** `target_x, target_y, target_z` + quaternione `hand_quat_q{x,y,z,w}`  
**Outputs:** `Root.001_ry, Root.002_rx, Root.003_rx, Root.004_rx, Head_ry`  
**Split:** `spatial_hash` per garantire generalizzazione spaziale  

In [1]:
import pandas as pd
import numpy as np
import os, json, time, hashlib
import torch
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

print(f'PyTorch {torch.__version__}, CUDA={torch.cuda.is_available()}')

PyTorch 2.11.0+cu130, CUDA=False


## ⚙️ Configuration

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────
CSV_PATH   = 'dataset_fixed.csv'
MODEL_DIR  = 'models/ik_model_research_spatial/xarm_quat'
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Split ─────────────────────────────────────────────────────────────────
SPLIT_MODE        = 'spatial_hash'  # 'random' or 'spatial_hash'
SPATIAL_CELL_SIZE = 0.05            # metres
VAL_SPLIT         = 0.1
TEST_SPLIT        = 0.1

# ── Training ──────────────────────────────────────────────────────────────
EPOCHS      = 120
BATCH_SIZE  = 256
HIDDEN      = [256, 256, 256, 128]
LR          = 5e-4
LOSS        = 'huber'          # 'huber' | 'mse' | 'mae'
WEIGHTING   = 'inverse_std'   # 'none'  | 'inverse_std'

# ── Columns ───────────────────────────────────────────────────────────────
QUAT_PREFIX  = 'hand_quat'    # → hand_quat_qx/qy/qz/qw
OUTPUT_COLS  = ['Root.001_ry', 'Root.002_rx', 'Root.003_rx', 'Root.004_rx', 'Head_ry']
SIN_COS      = False          # True → predict sin/cos encoded angles

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


## 📦 Load & Inspect Data

In [3]:
df = pd.read_csv(CSV_PATH)
print(f'{len(df):,} rows  |  {df.shape[1]} columns')
df.head(3)

21,741,411 rows  |  22 columns


,target_x,target_y,target_z,hand_quat_qx,hand_quat_qy,hand_quat_qz,hand_quat_qw,Root.001_ry,Root.002_rx,Root.003_rx,...,Root.001_ry_sin,Root.001_ry_cos,Root.002_rx_sin,Root.002_rx_cos,Root.003_rx_sin,Root.003_rx_cos,Root.004_rx_sin,Root.004_rx_cos,Head_ry_sin,Head_ry_cos
0,-0.2,-0.19397,-0.05,0.769304,0.565324,-0.294709,0.041548,-0.795495,1.428087,0.178582,...,-0.714210,0.699931,0.989834,0.142225,0.177634,0.984097,0.802290,0.596934,-0.881534,-0.472120
1,-0.2,-0.19397,-0.05,0.816882,0.494089,-0.289798,0.067798,-0.795627,1.428643,0.177629,...,-0.714303,0.699837,0.989913,0.141675,0.176697,0.984265,0.802526,0.596617,-0.951615,-0.307294
2,-0.2,-0.19397,-0.05,0.857884,0.418875,-0.282554,0.093503,-0.795760,1.429339,0.176359,...,-0.714395,0.699742,0.990012,0.140986,0.175446,0.984489,0.802868,0.596156,-0.991168,-0.132610


In [4]:
XYZ_COLS  = ['target_x', 'target_y', 'target_z']
QUAT_COLS = [f'{QUAT_PREFIX}_qx', f'{QUAT_PREFIX}_qy',
             f'{QUAT_PREFIX}_qz', f'{QUAT_PREFIX}_qw']
IN_COLS   = XYZ_COLS + QUAT_COLS

if SIN_COS:
    OUT_COLS = [c + s for c in OUTPUT_COLS for s in ['_sin', '_cos']]
else:
    OUT_COLS = OUTPUT_COLS

print('Input  cols:', IN_COLS)
print('Output cols:', OUT_COLS)

# sanity check
missing = [c for c in IN_COLS + OUT_COLS if c not in df.columns]
assert not missing, f'Missing columns: {missing}'

Input  cols: ['target_x', 'target_y', 'target_z', 'hand_quat_qx', 'hand_quat_qy', 'hand_quat_qz', 'hand_quat_qw']
Output cols: ['Root.001_ry', 'Root.002_rx', 'Root.003_rx', 'Root.004_rx', 'Head_ry']


## ✂️ Train / Val / Test Split

In [5]:
def spatial_hash_split(df, cell_size, val_frac, test_frac, seed=42):
    xs = (df['target_x'] / cell_size).astype(int)
    ys = (df['target_y'] / cell_size).astype(int)
    zs = (df['target_z'] / cell_size).astype(int)
    cell_keys = xs.astype(str) + '_' + ys.astype(str) + '_' + zs.astype(str)

    def cell_hash(k):
        return int(hashlib.md5((str(seed) + k).encode()).hexdigest(), 16) % 1000

    h = cell_keys.map(cell_hash)
    vt = int(val_frac  * 1000)
    tt = int(test_frac * 1000)
    return np.where(h < vt, 'val', np.where(h < vt + tt, 'test', 'train'))


if SPLIT_MODE == 'spatial_hash':
    split_labels = spatial_hash_split(df, SPATIAL_CELL_SIZE, VAL_SPLIT, TEST_SPLIT, SEED)
else:
    rng = np.random.default_rng(SEED)
    split_labels = np.full(len(df), 'train', dtype=object)
    idx = rng.permutation(len(df))
    nv  = int(len(df) * VAL_SPLIT)
    nt  = int(len(df) * TEST_SPLIT)
    split_labels[idx[:nv]]       = 'val'
    split_labels[idx[nv:nv+nt]]  = 'test'

m_tr = split_labels == 'train'
m_va = split_labels == 'val'
m_te = split_labels == 'test'
print(f'train={m_tr.sum():,}  val={m_va.sum():,}  test={m_te.sum():,}')

train=16,828,755  val=2,304,754  test=2,607,902


## 🔢 Normalisation & DataLoaders

In [6]:
X_all = df[IN_COLS].values.astype(np.float32)
Y_all = df[OUT_COLS].values.astype(np.float32)

X_mean = X_all[m_tr].mean(0); X_std = X_all[m_tr].std(0) + 1e-8
Y_mean = Y_all[m_tr].mean(0); Y_std = Y_all[m_tr].std(0) + 1e-8

X_norm = (X_all - X_mean) / X_std
Y_norm = (Y_all - Y_mean) / Y_std

# target weighting
if WEIGHTING == 'inverse_std':
    raw_std = Y_all[m_tr].std(0) + 1e-8
    weights = torch.tensor(1.0 / raw_std, dtype=torch.float32)
    weights = (weights / weights.mean()).to(device)
else:
    weights = torch.ones(len(OUT_COLS), dtype=torch.float32).to(device)

print('Input  mean:', np.round(X_mean, 4))
print('Output std :', np.round(Y_std,  4))
print('Weights    :', np.round(weights.cpu().numpy(), 4))

Input  mean: [-0.1246 -0.0239  0.0388  0.0819 -0.2607 -0.0049  0.2252]
Output std : [0.7727 0.5932 0.841  1.0721 0.9578]
Weights    : [1.0529 1.3715 0.9673 0.7588 0.8494]


In [7]:
class IKDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X)
        self.Y = torch.from_numpy(Y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.Y[i]

nw = min(4, os.cpu_count() or 1)
train_loader = DataLoader(IKDataset(X_norm[m_tr], Y_norm[m_tr]),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=nw, pin_memory=True)
val_loader   = DataLoader(IKDataset(X_norm[m_va], Y_norm[m_va]),
                          batch_size=BATCH_SIZE*4, shuffle=False, num_workers=nw, pin_memory=True)
test_loader  = DataLoader(IKDataset(X_norm[m_te], Y_norm[m_te]),
                          batch_size=BATCH_SIZE*4, shuffle=False, num_workers=nw, pin_memory=True)
print(f'Batches → train={len(train_loader)}  val={len(val_loader)}  test={len(test_loader)}')

Batches → train=65738  val=2251  test=2547


## 🧠 Model

In [8]:
class IKMLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.LayerNorm(h), nn.GELU()]
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

model = IKMLP(len(IN_COLS), len(OUT_COLS), HIDDEN).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nParametri totali: {total_params:,}')

IKMLP(
  (net): Sequential(
    (0): Linear(in_features=7, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): Linear(in_features=256, out_features=256, bias=True)
    (4): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (5): GELU(approximate='none')
    (6): Linear(in_features=256, out_features=256, bias=True)
    (7): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (8): GELU(approximate='none')
    (9): Linear(in_features=256, out_features=128, bias=True)
    (10): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (11): GELU(approximate='none')
    (12): Linear(in_features=128, out_features=5, bias=True)
  )
)

Parametri totali: 168,965


## 🏋️ Training

In [9]:
LOSS_FNS = {'huber': nn.HuberLoss(reduction='none'),
            'mse':   nn.MSELoss(reduction='none'),
            'mae':   nn.L1Loss(reduction='none')}
criterion  = LOSS_FNS[LOSS]
optimizer  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)

def weighted_loss(pred, target):
    return (criterion(pred, target) * weights.unsqueeze(0)).mean()

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()

    loop = tqdm(loader, leave=False)  # 👈 QUI

    with ctx:
        for X, Y in loop:
            X, Y = X.to(device), Y.to(device)

            if train:
                optimizer.zero_grad()

            pred = model(X)
            loss = weighted_loss(pred, Y)

            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total += loss.item() * len(X)

            loop.set_postfix(loss=loss.item())

    return total / len(loader.dataset)

best_val  = float('inf')
history   = []
ckpt_path = os.path.join(MODEL_DIR, 'best_model.pt')

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader,   train=False)
    scheduler.step()
    history.append({'epoch': epoch, 'train': tr, 'val': va})

    tag = ''
    if va < best_val:
        best_val = va
        torch.save(model.state_dict(), ckpt_path)
        tag = '  ✓'

    if epoch % 10 == 0 or epoch == 1:
        print(f'Ep {epoch:4d}/{EPOCHS} | train={tr:.5f}  val={va:.5f} | '
              f'{time.time()-t0:.1f}s{tag}')

print(f'\nBest val loss: {best_val:.5f}')

  0%|                                                 | 0/65738 [00:00<?, ?it/s]/home/marioboss/Desktop/csv belli/marioenv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
                                                                                

Ep    1/120 | train=0.00005  val=0.00001 | 668.8s  ✓


Ep   10/120 | train=0.00000  val=0.00000 | 592.8s


KeyboardInterrupt: 

## 📊 Evaluation sul Test Set

In [ ]:
model.load_state_dict(torch.load(ckpt_path, map_location=device))
test_loss = run_epoch(test_loader, train=False)
print(f'Test loss ({LOSS}, weighted): {test_loss:.5f}\n')

all_pred, all_true = [], []
model.eval()
with torch.no_grad():
    for X, Y in test_loader:
        all_pred.append(model(X.to(device)).cpu())
        all_true.append(Y)

preds   = torch.cat(all_pred).numpy()
targets = torch.cat(all_true).numpy()

# de-normalise per valutare in radianti
if not SIN_COS:
    preds_raw   = preds   * Y_std + Y_mean
    targets_raw = targets * Y_std + Y_mean
    mae_per = np.abs(preds_raw - targets_raw).mean(axis=0)
    print('MAE per joint (radianti):')
    for col, m in zip(OUT_COLS, mae_per):
        print(f'  {col:20s}: {m:.5f} rad  ({np.degrees(m):.3f}°)')
else:
    mae_per = np.abs(preds - targets).mean(axis=0)
    print('MAE per colonna sin/cos (spazio normalizzato):')
    for col, m in zip(OUT_COLS, mae_per):
        print(f'  {col:30s}: {m:.5f}')

## 📈 Loss Curve

In [ ]:
import matplotlib.pyplot as plt
hist_df = pd.DataFrame(history)
plt.figure(figsize=(10, 4))
plt.plot(hist_df['epoch'], hist_df['train'], label='train')
plt.plot(hist_df['epoch'], hist_df['val'],   label='val')
plt.xlabel('Epoch'); plt.ylabel('Weighted Loss'); plt.legend(); plt.grid(True)
plt.title('Training History')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'loss_curve.png'), dpi=120)
plt.show()

## 💾 Salva Artefatti

In [ ]:
np.save(os.path.join(MODEL_DIR, 'X_mean.npy'), X_mean)
np.save(os.path.join(MODEL_DIR, 'X_std.npy'),  X_std)
np.save(os.path.join(MODEL_DIR, 'Y_mean.npy'), Y_mean)
np.save(os.path.join(MODEL_DIR, 'Y_std.npy'),  Y_std)

meta = {
    'in_cols': IN_COLS, 'out_cols': OUT_COLS, 'hidden': HIDDEN,
    'loss': LOSS, 'target_weighting': WEIGHTING,
    'quat_input_prefix': QUAT_PREFIX,
    'split_mode': SPLIT_MODE, 'spatial_cell_size': SPATIAL_CELL_SIZE,
    'epochs': EPOCHS, 'best_val_loss': best_val, 'test_loss': float(test_loss),
    'sin_cos_targets': SIN_COS, 'seed': SEED
}
with open(os.path.join(MODEL_DIR, 'meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)

hist_df.to_csv(os.path.join(MODEL_DIR, 'training_history.csv'), index=False)

print(f'Artefatti salvati in: {MODEL_DIR}')
print('  best_model.pt, X/Y_mean/std.npy, meta.json, training_history.csv, loss_curve.png')

## 🤖 Inferenza (esempio)
Come usare il modello salvato per predire le joint angles da una nuova posa target.

In [ ]:
def load_model(model_dir):
    with open(os.path.join(model_dir, 'meta.json')) as f:
        meta = json.load(f)
    m = IKMLP(len(meta['in_cols']), len(meta['out_cols']), meta['hidden'])
    m.load_state_dict(torch.load(os.path.join(model_dir, 'best_model.pt'), map_location='cpu'))
    m.eval()
    xm = np.load(os.path.join(model_dir, 'X_mean.npy'))
    xs = np.load(os.path.join(model_dir, 'X_std.npy'))
    ym = np.load(os.path.join(model_dir, 'Y_mean.npy'))
    ys = np.load(os.path.join(model_dir, 'Y_std.npy'))
    return m, meta, xm, xs, ym, ys

def predict_ik(model, xm, xs, ym, ys, target_xyz, quaternion_xyzw):
    """target_xyz: [x,y,z]  quaternion_xyzw: [qx,qy,qz,qw]"""
    inp = np.array(target_xyz + quaternion_xyzw, dtype=np.float32)
    inp_norm = (inp - xm) / xs
    with torch.no_grad():
        out_norm = model(torch.from_numpy(inp_norm).unsqueeze(0)).numpy()[0]
    return out_norm * ys + ym

# Esempio
mdl, meta_, xm_, xs_, ym_, ys_ = load_model(MODEL_DIR)
joints = predict_ik(mdl, xm_, xs_, ym_, ys_,
                    target_xyz=[-0.15, 0.10, 0.03],
                    quaternion_xyzw=[0.58, -0.06, -0.04, 0.21])
print('Predicted joint angles (rad):')
for col, val in zip(meta_['out_cols'], joints):
    print(f'  {col:20s}: {val:.4f}')